# Introduction to Large Language Models (LLMs)

## Exercise 1: What are Large Language Models (LLMs)?

**What are LLMs?**

Large Language Models are deep neural networks, usually built on the Transformer architecture, that are trained on massive amounts of text data to learn the statistical patterns of human language. They are designed to predict, generate, and understand text by learning relationships between words, phrases, and broader context across very long sequences. Because they are trained on such large and diverse datasets, LLMs can perform a wide range of language tasks — answering questions, summarizing text, translating languages, writing code, and holding conversations — often without being explicitly trained for each individual task. Their core design purpose is general-purpose language understanding and generation, which can then be adapted or specialized for more specific applications.

**Setup**

In [ ]:
# Install necessary libraries
!pip install transformers matplotlib --quiet

# Import required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

**Loading a pretrained model and tokenizer**

In [ ]:
# 2. Loading a pretrained model and tokenizer
model_name = "gpt2"  # GPT-2 is used here for demonstration; can be replaced with models like "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence. 
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")

## Exercise 2: Transformer Architecture and Tokenization

**What is tokenization?**

Tokenization is the process of breaking down raw text into smaller units called tokens, which can be whole words, subwords, or even individual characters, depending on the tokenizer's vocabulary. Transformer models cannot process raw text directly — they operate on numbers — so each token is mapped to a unique integer ID from the model's vocabulary. This step is essential because it converts unstructured human language into a structured numerical format that the neural network can process through its embedding layers. Subword tokenization (used by GPT-2) is particularly useful because it can represent rare or unknown words by breaking them into smaller, more common subword pieces, rather than treating every unfamiliar word as a single unknown token.

In [ ]:
text = "Machine learning models can understand and generate human language."

In [ ]:
# 2. Tokenize input text
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")

In [ ]:
# 3. Visualizing the tokenization process
plt.figure(figsize=(10, 4))
plt.bar(tokens, token_ids, color="skyblue")
plt.xlabel("Tokens")
plt.ylabel("Token ID")
plt.title("Tokenization of Input Text \u2014 Token vs Token ID")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Exercise 3: Understanding Token IDs and Special Prefixes

In [ ]:
# 1. Print the ID of each token from `text`
print(f"Text: {text}\n")
for token, token_id in zip(tokens, token_ids):
    print(f"Token: {token!r:20s} -> ID: {token_id}")

**What does the special prefix 'G with dot above' (\u0120) indicate?**

GPT-2 uses a Byte-Pair Encoding (BPE) tokenizer where a special character (often displayed as 'G with a dot above it', the byte-level representation of a literal space) is prepended to tokens that are preceded by a whitespace in the original text. In other words, this marked token means "a space followed by this subword", distinguishing it from the same subword appearing at the very start of a sequence or attached to a previous token without a space (e.g., as part of a compound word). This prefix allows the tokenizer to preserve information about word boundaries and spacing, which is necessary for the model to later reconstruct text with correct spacing when generating output, since the tokenizer must remain fully reversible (detokenization should recover the exact original text, including whitespace).

## Exercise 4: Pretraining vs. Fine-Tuning

**Pretraining**

Pretraining is the initial, large-scale training phase where a Transformer model learns general language patterns from a massive, diverse corpus of text (books, websites, articles, code, etc.), usually without any task-specific labels. The model is typically trained on a self-supervised objective, such as predicting the next word in a sequence (causal language modeling, used by GPT-2) or predicting masked words within a sentence (masked language modeling, used by BERT). This phase is extremely computationally expensive and requires huge datasets, but it produces a model with broad, general-purpose language understanding that captures grammar, facts, reasoning patterns, and even some world knowledge.

**Fine-Tuning**

Fine-tuning is a second, much smaller training phase where the already-pretrained model is further trained on a smaller, task-specific or domain-specific dataset. This adapts the model's general language abilities to a particular use case, such as sentiment classification, medical text summarization, or customer support chat. Because the model already has strong general language understanding from pretraining, fine-tuning requires far less data and computational resources than training a model from scratch, and it allows a single pretrained model to be specialized into many different downstream applications.

## Exercise 5: Generate Simple Text

In [ ]:
input_text = "The future of artificial intelligence is"

In [ ]:
input_ids = tokenizer.encode(input_text, return_tensors="pt")

output_ids = model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    pad_token_id=tokenizer.eos_token_id
)

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Input: {input_text}")
print(f"Generated Output: {output_text}")

The model generates text by predicting the most probable next token given the input prompt, then appending that token to the sequence and repeating the process — each new token is predicted based on all previous tokens (prompt + already-generated tokens). This autoregressive, sequential generation process continues until either the maximum length is reached or the model produces a special end-of-sequence token.